# Biased-generalization analysis (DiffiT 256²)

Interactive version of `experiments/analyze_sample_split.py`. Loads matching
checkpoints from the two split-A / split-B training runs, computes the
sample-split cosine distance with matched noise, reads the already-logged
`Loss/test` from each run's `stats.jsonl`, and plots the dual-axis
biased-generalization figure with Plotly.

**Shape of the plot:**
- Left axis (blue): cosine distance between images generated by model A and B under identical noise.
- Right axis (red/orange): DSM test loss on the held-out val split, per model.
- Dotted verticals mark the minima of each curve.

The biased-generalization window is the interval between the blue minimum (cosine distance bottoms out) and the red minimum (test loss bottoms out).

## 1 — Imports and path setup

In [ ]:
import os, sys, json, time
from pathlib import Path

# Make the repo importable — this notebook lives in experiments/notebooks/
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent.parent          # .../DiffiT-v2
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))

import numpy as np
import torch
from tqdm.auto import tqdm
import plotly.graph_objects as go
from diffusers.models import AutoencoderKL

# Functions from the analysis script
from analyze_sample_split import (
    match_checkpoints,
    load_model,
    cosine_distance,
    read_test_loss,
)
from diffit import create_diffusion, diffusion_defaults

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

## 2 — Configuration

Adjust `RUN_A` / `RUN_B` to point at your split-A and split-B run directories. The directory listing auto-detects which run is which based on the `splitA`/`splitB` suffix.

In [ ]:
RUNS_DIR = REPO_ROOT / 'experiments' / 'runs' / '256'

# Auto-detect A/B from dirname suffix.
run_dirs = {d.name: d for d in RUNS_DIR.iterdir() if d.is_dir()}
RUN_A = next((str(p) for n, p in run_dirs.items() if 'splitA' in n), None)
RUN_B = next((str(p) for n, p in run_dirs.items() if 'splitB' in n), None)
assert RUN_A and RUN_B, f'Could not find splitA/splitB in {RUNS_DIR}: {list(run_dirs.keys())}'

# --- Analysis knobs --------------------------------------------------------
NUM_SAMPLES     = 64      # images per checkpoint for cosine distance
BATCH_SIZE      = 16
NUM_STEPS       = 50      # DDPM sampling steps per image
MAX_CHECKPOINTS = 20      # set to None for all; otherwise evenly subsampled
BASE_SEED       = 0

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f'RUN_A: {RUN_A}')
print(f'RUN_B: {RUN_B}')
print(f'device: {DEVICE}, amp: {AMP_DTYPE}')

## 3 — Discover matching checkpoints and read test loss

In [ ]:
pairs = match_checkpoints(RUN_A, RUN_B)
print(f'Found {len(pairs)} matching checkpoint pairs')

# Optional subsample for a faster first pass.
if MAX_CHECKPOINTS is not None and len(pairs) > MAX_CHECKPOINTS:
    idx = np.linspace(0, len(pairs) - 1, MAX_CHECKPOINTS).round().astype(int)
    pairs = [pairs[i] for i in idx]
    print(f'Subsampled to {len(pairs)} checkpoints evenly across the run')

print('kimgs to analyze:', [k for k, _, _ in pairs])

test_a = read_test_loss(RUN_A)
test_b = read_test_loss(RUN_B)
print(f'Test-loss scalars: A={len(test_a)} points, B={len(test_b)} points')

## 4 — Load the VAE decoder (once)

In [ ]:
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-ema').to(DEVICE).eval()
for p in vae.parameters():
    p.requires_grad_(False)

diffusion = create_diffusion(**diffusion_defaults())
print('VAE + diffusion schedule ready.')

## 5 — Run the sample-split analysis

For each `(A_i, B_i)` checkpoint pair:
1. Load both EMA models to `DEVICE`.
2. Draw `NUM_SAMPLES` matched-noise image pairs (identical `z_T`, identical per-step noise, identical class labels).
3. Decode via the VAE and measure mean cosine distance over the batch.
4. Free GPU memory and move on.

This is the bulk of the wall-clock time; expect **~30–60 s per checkpoint at NUM_SAMPLES=64, NUM_STEPS=50 on an H200**. Scale accordingly for weaker GPUs.

In [ ]:
results = []
for kimg, path_a, path_b in tqdm(pairs, desc='checkpoints'):
    t0 = time.time()

    model_a, image_size, num_classes_a = load_model(path_a, DEVICE)
    model_b, _, num_classes_b          = load_model(path_b, DEVICE)
    assert num_classes_a == num_classes_b, f'Class count mismatch: {num_classes_a} vs {num_classes_b}'
    latent_size = image_size // 8

    mean, sem = cosine_distance(
        model_a, model_b, vae, diffusion, DEVICE,
        num_samples=NUM_SAMPLES,
        latent_size=latent_size,
        batch_size=BATCH_SIZE,
        num_steps=NUM_STEPS,
        num_classes=num_classes_a,
        base_seed=BASE_SEED,
    )

    results.append({
        'kimg': kimg,
        'cosine_distance': mean,
        'cosine_distance_sem': sem,
        'test_loss_a': test_a.get(kimg, float('nan')),
        'test_loss_b': test_b.get(kimg, float('nan')),
    })

    # Free GPU before next iteration
    del model_a, model_b
    torch.cuda.empty_cache()

    print(f'  kimg={kimg:>6d}  cos={mean:.4f}±{sem:.4f}  '
          f'tl_A={results[-1]["test_loss_a"]:.4f}  '
          f'tl_B={results[-1]["test_loss_b"]:.4f}  '
          f'({time.time() - t0:.1f}s)')

print(f'\nDone: {len(results)} checkpoint pairs analyzed.')

In [ ]:
results

## 6 — Plot (Plotly, dual Y-axis)

In [ ]:
import json, math, pathlib

RESULTS_PATH = pathlib.Path.cwd().parent.parent / 'experiments' / 'analysis' / '256' / 'results.json'
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

def _clean(r):
    return {k: (None if isinstance(v, float) and math.isnan(v) else v)
            for k, v in r.items()}

with open(RESULTS_PATH, 'w') as f:
    json.dump([_clean(r) for r in results], f, indent=2)

print(f'Saved {len(results)} records → {RESULTS_PATH}')


In [ ]:
import json, pathlib
RESULTS_PATH = pathlib.Path.cwd().parent.parent / 'experiments' / 'analysis' / '256' / 'results.json'
with open(RESULTS_PATH) as f:
    results = json.load(f)

# Turn None back into NaN for the plot code
import math
for r in results:
    for k in ('test_loss_a', 'test_loss_b'):
        if r.get(k) is None:
            r[k] = float('nan')

print(f'Loaded {len(results)} records')


In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# ── helpers ────────────────────────────────────────────────────────────
def read_tb_scalar(run_dir, tag):
    ea = EventAccumulator(run_dir, size_guidance={'scalars': 0}); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.empty(0), np.empty(0)
    s = ea.Scalars(tag)
    return (np.fromiter((e.step  for e in s), float),
            np.fromiter((e.value for e in s), float))

def ema(y, a=0.05):
    if len(y) == 0: return y
    out = np.empty_like(y, float); acc = y[0]
    for i, v in enumerate(y):
        acc = a * v + (1 - a) * acc; out[i] = acc
    return out

# ── gather all series ──────────────────────────────────────────────────
RUNS   = {'A': RUN_A, 'B': RUN_B}
tb     = {s: {tag: read_tb_scalar(r, tag) for tag in ('Loss/train', 'Metrics/FID')}
          for s, r in RUNS.items()}

kimgs   = np.array([r['kimg']                for r in results])
# Flip: plot cosine SIMILARITY (1 = identical, 0 = orthogonal) instead of distance.
# Because A and B share initialization, similarity starts at ~1 and drops through
# the biased phase — giving a valley-shaped curve like the paper's Fig 1(a).
cos     = 1 - np.array([r['cosine_distance']     for r in results])
cos_err =     np.array([r['cosine_distance_sem'] for r in results])  # SEM unchanged under (1-x)


# ── style ──────────────────────────────────────────────────────────────
COLOR_COS = '#1f77b4'
COLORS    = {'A': '#d62728', 'B': '#ff7f0e'}
DOMAIN_R  = 0.88   # leave 12% of x-width for the second right-axis

# ── figure ─────────────────────────────────────────────────────────────
fig = go.Figure()

# left axis (y): cosine distance
fig.add_trace(go.Scatter(
    x=kimgs, y=cos, error_y=dict(type='data', array=cos_err),
    mode='lines+markers', name='cosine similarity',
    line=dict(color=COLOR_COS, width=3), marker=dict(size=9),
    yaxis='y',
))

# right-inner axis (y2): train + test loss
for s in ('A', 'B'):
    kx, ky = tb[s]['Loss/train']
    fig.add_trace(go.Scatter(
        x=kx, y=ema(ky), mode='lines',
        name=f'train loss ({s})',
        line=dict(color=COLORS[s], width=1.5), yaxis='y2',
    ))
    fig.add_trace(go.Scatter(
        x=kimgs, y=test[s], mode='lines+markers',
        name=f'test loss ({s})',
        line=dict(color=COLORS[s], width=2, dash='dash'),
        marker=dict(size=6, symbol='x'), yaxis='y2',
    ))

# right-outer axis (y3, log): FID
for s in ('A', 'B'):
    fx, fy = tb[s]['Metrics/FID']
    fig.add_trace(go.Scatter(
        x=fx, y=fy, mode='lines+markers',
        name=f'FID ({s})',
        line=dict(color=COLORS[s], width=2, dash='dot'),
        marker=dict(size=7, symbol='diamond'), yaxis='y3',
    ))

# vertical markers
if len(cos) > 1:
    fig.add_vline(x=float(kimgs[np.nanargmin(cos)]),
                  line=dict(color=COLOR_COS, dash='dot', width=1.5))
tl_mean = np.nanmean(np.stack([test['A'], test['B']]), axis=0)
if not np.all(np.isnan(tl_mean)):
    fig.add_vline(x=float(kimgs[np.nanargmin(tl_mean)]),
                  line=dict(color='#666', dash='dot', width=1.5))

# layout with three y-axes
fig.update_layout(
    title='Biased generalization in DiffiT (256²) — cosine / losses / FID',
    template='plotly_white',
    hovermode='x unified',
    height=600, width=1150,
    xaxis=dict(title='kimg', domain=[0.0, DOMAIN_R]),
    yaxis =dict(title=dict(text='Sample-split cosine similarity',
                           font=dict(color=COLOR_COS)),
                tickfont=dict(color=COLOR_COS)),
    yaxis2=dict(title='DSM loss (train / test)',
                overlaying='y', side='right'),
    yaxis3=dict(title='FID (log)',
                overlaying='y', side='right', anchor='free',
                position=DOMAIN_R + 0.08, type='log'),
    legend=dict(orientation='v', x=1.18, y=0.5,
                bgcolor='rgba(255,255,255,0.85)'),
)

fig.show()


## 7 — Save raw results and figure

In [ ]:
OUTDIR = REPO_ROOT / 'experiments' / 'analysis' / '256'
OUTDIR.mkdir(parents=True, exist_ok=True)

with open(OUTDIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)
fig.write_html(str(OUTDIR / 'figure1a.html'))
fig.write_image(str(OUTDIR / 'figure1a.png'), scale=2)  # requires kaleido

print(f'Saved:\n  {OUTDIR / "results.json"}\n  {OUTDIR / "figure1a.html"}\n  {OUTDIR / "figure1a.png"}')

## 8 — Interpretation

Read the plot as follows:

- **Blue minimum ≪ red minimum**: biased-generalization phase present. The gap is your "biased window" — in that span, both models decrease test loss while drifting toward their individual training samples under matched noise.
- **Blue and red minima coincide**: no measurable biased-generalization phase under this budget.
- **Blue still monotonically decreasing at end of training**: train longer; you haven't hit the U-turn yet.

For a paper-quality figure you'd want `NUM_SAMPLES ≥ 256` and all checkpoints (`MAX_CHECKPOINTS = None`). The numbers above are tuned for in-notebook iteration speed.